In [1]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [2]:
builder = (SparkSession.builder
           .appName("checkpoints")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "512m")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark-catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder, ['org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1']).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-af4de3cd-51de-4543-96c0-db63a4762941;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.1 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in centra

In [3]:
df = (spark.readStream
      .format("kafka")
      .option("kafka.bootstrap.servers", "kafka:9092")
      .option("subscribe", "users")
      .option("startingOffsets", "earliest")
      .load())

In [4]:
schema = StructType([
    StructField('id', IntegerType(), True),
    StructField('name', StringType(), True),
    StructField('age', IntegerType(), True),
    StructField('gender', StringType(), True),
    StructField('country', StringType(), True)])

df = df.withColumn('value', from_json(col('value').cast("STRING"), schema))

In [5]:
df = df.select(
    col('value.id').alias('id'),
    col('value.name').alias('name'),
    col('value.age').alias('age'),
    col('value.gender').alias('gender'),
    col('value.country').alias('country'))

In [6]:
query = (df.writeStream
         .format('console')
         .outputMode('append')
         .start())

-------------------------------------------
Batch: 0
-------------------------------------------
+---+------+---+------+---------+
| id|  name|age|gender|  country|
+---+------+---+------+---------+
|  7|user66| 31|     M|      USA|
| 68| user1| 47|     M|    China|
|  9|user74| 30|     F|      USA|
| 66|user21| 43|     M|       UK|
| 22|user25| 64|     F|  Germany|
| 96|user89| 21|     M|  Moldova|
| 46|user79| 38|     F|      USA|
| 25|user57| 45|     F|      USA|
|  4|user47| 29|     M|Australia|
| 91| user3| 57|     M|  Moldova|
| 73|user21| 46|     M|Australia|
| 58| user5| 53|     F|  Germany|
| 31|user82| 27|     M|      USA|
|  9|user38| 36|     M|      USA|
| 14|user36| 59|     M|Australia|
| 20|user59| 60|     M|      USA|
| 55|user29| 25|     F|       UK|
| 20|user27| 56|     M|  Germany|
| 55| user3| 64|     M|   Canada|
| 33|user26| 61|     F|       UK|
+---+------+---+------+---------+
only showing top 20 rows



-------------------------------------------
Batch: 1
-------------------------------------------
+---+-----+---+------+-------+
| id| name|age|gender|country|
+---+-----+---+------+-------+
| 13|user9| 29|     M|    USA|
+---+-----+---+------+-------+



In [7]:
query.stop()

25/06/03 08:58:00 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 2, writer: ConsoleWriter[numRows=20, truncate=true]] is aborting.
25/06/03 08:58:00 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 2, writer: ConsoleWriter[numRows=20, truncate=true]] aborted.


In [8]:
query = (df.writeStream
         .format('console')
         .outputMode('append')
         .trigger(processingTime='30 seconds')
         .start())

-------------------------------------------
Batch: 0
-------------------------------------------
+---+------+---+------+---------+
| id|  name|age|gender|  country|
+---+------+---+------+---------+
|  7|user66| 31|     M|      USA|
| 68| user1| 47|     M|    China|
|  9|user74| 30|     F|      USA|
| 66|user21| 43|     M|       UK|
| 22|user25| 64|     F|  Germany|
| 96|user89| 21|     M|  Moldova|
| 46|user79| 38|     F|      USA|
| 25|user57| 45|     F|      USA|
|  4|user47| 29|     M|Australia|
| 91| user3| 57|     M|  Moldova|
| 73|user21| 46|     M|Australia|
| 58| user5| 53|     F|  Germany|
| 31|user82| 27|     M|      USA|
|  9|user38| 36|     M|      USA|
| 14|user36| 59|     M|Australia|
| 20|user59| 60|     M|      USA|
| 55|user29| 25|     F|       UK|
| 20|user27| 56|     M|  Germany|
| 55| user3| 64|     M|   Canada|
| 33|user26| 61|     F|       UK|
+---+------+---+------+---------+
only showing top 20 rows



-------------------------------------------
Batch: 1
-------------------------------------------
+---+------+---+------+-------+
| id|  name|age|gender|country|
+---+------+---+------+-------+
| 68|user36| 46|     F|Germany|
| 51|user18| 21|     M|    USA|
| 23|user49| 34|     M|Moldova|
+---+------+---+------+-------+

-------------------------------------------
Batch: 2
-------------------------------------------
+---+------+---+------+---------+
| id|  name|age|gender|  country|
+---+------+---+------+---------+
| 50| user2| 56|     F|  Moldova|
| 46|user88| 42|     F|   Canada|
| 73|user12| 55|     M|Australia|
+---+------+---+------+---------+



In [9]:
query.stop()

In [10]:
query = (df.writeStream
         .format('console')
         .outputMode('append')
         .trigger(once=True)
         .start())

-------------------------------------------
Batch: 0
-------------------------------------------
+---+------+---+------+---------+
| id|  name|age|gender|  country|
+---+------+---+------+---------+
|  7|user66| 31|     M|      USA|
| 68| user1| 47|     M|    China|
|  9|user74| 30|     F|      USA|
| 66|user21| 43|     M|       UK|
| 22|user25| 64|     F|  Germany|
| 96|user89| 21|     M|  Moldova|
| 46|user79| 38|     F|      USA|
| 25|user57| 45|     F|      USA|
|  4|user47| 29|     M|Australia|
| 91| user3| 57|     M|  Moldova|
| 73|user21| 46|     M|Australia|
| 58| user5| 53|     F|  Germany|
| 31|user82| 27|     M|      USA|
|  9|user38| 36|     M|      USA|
| 14|user36| 59|     M|Australia|
| 20|user59| 60|     M|      USA|
| 55|user29| 25|     F|       UK|
| 20|user27| 56|     M|  Germany|
| 55| user3| 64|     M|   Canada|
| 33|user26| 61|     F|       UK|
+---+------+---+------+---------+
only showing top 20 rows



In [11]:
query.stop()

In [12]:
spark.stop()